# ⚾ 맞은 구종의 비중(구사율)이 회피 효과를 조절하는가 — 전체 표본 상호작용 검정

* **작성자:** DY
* **작성일:** 2026-10-05

## 1. 배경: 노트북 10의 질문을, 표본을 버리지 않고 다시 묻기

노트북 10은 "맞은 구종이 주무기(가장 많이 던지는 비-패스트볼 구종)였던 경우"로 표본을 좁혀서
(27,245건 → 6,083건), 반사실 예측이 더 신뢰할 만해지면 회피 효과가 더 크게 나오는지 봤습니다.
결과는 반대였습니다 — 효과가 사라졌습니다(+0.0003, p=0.47).

이게 표본이 작아져서 생긴 착시인지 확인해 봤습니다: 노트북 10 표본의 정밀도(SE=0.00047)로 노트북
09의 효과 크기(+0.0023)를 찾아낼 **검정력은 99.83%**였습니다. 즉 진짜 효과가 그만큼 컸다면 거의
확실히 잡혔을 것이고, 실제로는 훨씬 작게 나왔습니다 — 표본 크기 문제가 아니라는 뜻입니다.

그래도 "주무기 여부"를 이분법으로 자른 것 자체가 비효율적일 수 있습니다. 이 노트북은 노트북 09의
표본을 "주무기 1건만" 식으로 좁히지 않고, "맞은 구종이 그 투수에게 얼마나 중요했는가"를
**연속변수**(`hit_pitch_season_usage_rate`, 그 구종의 그 시즌 구사율)로 모델에 넣어, `avoided`
효과가 이 구사율에 따라 달라지는지를 **상호작용항**으로 직접 검정합니다. 노트북 10과 같은 이유로
맞은 구종이 패스트볼 계열이었던 경우는 제외하지만(3절), 그 외에는 표본을 하나도 더 버리지 않습니다 —
이분법적 부분표본보다 통계 효율이 높고(남은 표본의 검정력을 그대로 씀), "주무기일수록 회피 효과가
커지는가"라는 노트북 10의 질문에 더 정밀하게 답할 수 있습니다.

$$\text{selection\_value\_pa\_mean} \sim \text{avoided} \times \text{hit\_pitch\_season\_usage\_rate}
+ (\text{공변량들}) + (1 \mid \text{pitcher})$$

이 노트북은 **노트북 09를 수정하지 않습니다** — 09의 결과는 그대로 유지한 채, 이 상호작용 질문만
별도로 답합니다.

In [1]:
# 1. 경로 설정 (src 폴더 접근용)
import sys, os
sys.path.append(os.path.abspath('..'))

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from IPython.display import display
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import r2_score

from src.analysis.avoidance_stats import compute_season_usage_rate
from src.analysis.glmer_runner import (
    check_convergence,
    extract_lmer_fixed_effects,
    extract_variance_components,
    fit_lmer,
)
from src.analysis.mixed_effects_model import compute_icc_gaussian
from src.analysis.run_same_batter_rematch_analysis import build_xbh_rematch
from src.preprocessing.pitch_family import map_pitch_family

pd.set_option('display.width', 220)
pd.set_option('display.max_columns', 60)

TRAIN_SEASONS = (2021, 2022, 2023, 2024, 2025)
VALID_SEASON = 2026
RANDOM_STATE = 20261003

Error importing in API mode: ImportError("dlopen(/Users/dongyunkwak/GitHub/9th-first-project-baseball-2/.venv/lib/python3.14/site-packages/_rinterface_cffi_api.abi3.so, 0x0002): Library not loaded: /Library/Frameworks/R.framework/Versions/4.6/Resources/lib/libRblas.dylib\n  Referenced from: <9F0E20C3-D782-31B5-943E-66F62BCE77E8> /Users/dongyunkwak/GitHub/9th-first-project-baseball-2/.venv/lib/python3.14/site-packages/_rinterface_cffi_api.abi3.so\n  Reason: tried: '/Library/Frameworks/R.framework/Versions/4.6/Resources/lib/libRblas.dylib' (no such file), '/System/Volumes/Preboot/Cryptexes/OS/Library/Frameworks/R.framework/Versions/4.6/Resources/lib/libRblas.dylib' (no such file), '/Library/Frameworks/R.framework/Versions/4.6/Resources/lib/libRblas.dylib' (no such file)")


Trying to import in ABI mode.


## 2. 데이터 준비와 선택가치 계산 (노트북 09와 동일)

팀 CSV 로딩부터 "타석 전체로 집계한 선택가치"(`selection_value_pa_mean`)까지는 노트북 09와 완전히
같은 과정이고, 표본도 그대로 27,245건을 씁니다 — 이번엔 표본을 자르지 않습니다.

In [2]:
from src.preprocessing.research_sample import KEY_COLUMNS, list_research_csvs
from src.preprocessing.build_next_ab_dataset import EXTRA_BASE_HIT_EVENTS

LOAD_COLUMNS = [
    'game_pk', 'game_date', 'at_bat_number', 'pitch_number', 'pitcher', 'player_name', 'batter',
    'stand', 'p_throws', 'pitch_type', 'events', 'balls', 'strikes', 'outs_when_up',
    'inning', 'inning_topbot', 'on_1b', 'on_2b', 'on_3b', 'home_score', 'away_score',
    'woba_value', 'woba_denom',
]


def load_team_csv() -> pd.DataFrame:
    frames = [
        pd.read_csv(path, usecols=LOAD_COLUMNS, encoding='utf-8-sig', low_memory=False)
        for path in list_research_csvs()
    ]
    pitches = pd.concat(frames, ignore_index=True)
    pitches['season'] = pd.to_datetime(pitches['game_date']).dt.year
    return pitches


pitches = load_team_csv()
assert len(pitches) == 3_592_302 and pitches['pitcher'].nunique() == 1_067
assert not pitches.duplicated(KEY_COLUMNS).any()
print(f'{len(pitches):,}행, 투수 {pitches["pitcher"].nunique():,}명')

3,592,302행, 투수 1,067명


In [3]:
def add_situation_columns(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    on1, on2, on3 = out['on_1b'].notna(), out['on_2b'].notna(), out['on_3b'].notna()
    out['runners_n'] = on1.astype(int) + on2.astype(int) + on3.astype(int)
    out['risp'] = (on2 | on3).astype(int)
    out['score_diff'] = np.where(
        out['inning_topbot'] == 'Top', out['home_score'] - out['away_score'], out['away_score'] - out['home_score']
    )
    out['platoon_match'] = np.where(
        out['stand'].isna() | out['p_throws'].isna(), np.nan, (out['stand'] == out['p_throws']).astype(float)
    )
    return out


_t = pd.DataFrame({
    'on_1b': [np.nan, 1.0, np.nan], 'on_2b': [np.nan, np.nan, 2.0], 'on_3b': [np.nan, np.nan, np.nan],
    'inning_topbot': ['Top', 'Bot', 'Top'], 'home_score': [3, 5, 2], 'away_score': [3, 2, 4],
    'stand': ['L', 'R', 'R'], 'p_throws': ['R', 'R', 'L'],
})
_r = add_situation_columns(_t)
assert _r['runners_n'].tolist() == [0, 1, 1]
assert _r['risp'].tolist() == [0, 0, 1]
assert _r['score_diff'].tolist() == [0, -3, -2]
assert _r['platoon_match'].tolist() == [0.0, 1.0, 0.0]
print('add_situation_columns 테스트 통과')

pitches = add_situation_columns(pitches)

add_situation_columns 테스트 통과


In [4]:
season_usage = compute_season_usage_rate(pitches)
xbh = build_xbh_rematch(pitches, season_usage)
xbh = xbh[xbh['has_next_ab']].copy()
assert len(xbh) == 27_391, len(xbh)
print(f'재대결 이벤트 {len(xbh):,}건')

2026-10-06 12:17:51,193 [INFO] 장타 70319건 중 같은 타자·같은 투수 재대결 27391건 (39.0%)
        xbh_total  same_batter_rematch  rematch_share
season                                               
2021        12709                 4688         0.3689
2022        12186                 4623         0.3794
2023        13137                 5280         0.4019
2024        12427                 4977         0.4005
2025        12394                 4916         0.3966
2026         7466                 2907         0.3894


재대결 이벤트 27,391건


In [5]:
MODEL_FEATURES = [
    'pitch_type', 'stand', 'p_throws', 'balls', 'strikes', 'outs_when_up', 'runners_n', 'risp', 'score_diff',
    'season_usage_rate',
]
CATEGORICAL_FEATURES = ['pitch_type', 'stand', 'p_throws']


def build_decisive_pitch_table(pitches_sit: pd.DataFrame, season_usage_: pd.DataFrame) -> pd.DataFrame:
    decisive = pitches_sit[pitches_sit['woba_value'].notna() & pitches_sit['pitch_type'].notna()].copy()
    decisive = decisive.merge(
        season_usage_[['pitcher', 'season', 'pitch_type', 'season_usage_rate']],
        on=['pitcher', 'season', 'pitch_type'], how='left',
    )
    for col in CATEGORICAL_FEATURES:
        decisive[col] = decisive[col].astype('category')
    return decisive


decisive_table = build_decisive_pitch_table(pitches, season_usage)
assert decisive_table['season_usage_rate'].notna().mean() > 0.99
print(f'결정구 {len(decisive_table):,}건, season_usage_rate 결측 {decisive_table["season_usage_rate"].isna().sum()}건')

결정구 917,717건, season_usage_rate 결측 0건


In [6]:
def fit_run_value_model(table: pd.DataFrame, seasons) -> HistGradientBoostingRegressor:
    train = table[table['season'].isin(seasons)]
    model = HistGradientBoostingRegressor(
        categorical_features='from_dtype', random_state=RANDOM_STATE, max_iter=200, early_stopping=True,
    )
    model.fit(train[MODEL_FEATURES], train['woba_value'])
    return model


train_set = decisive_table[decisive_table['season'].isin(TRAIN_SEASONS)]
valid_set = decisive_table[decisive_table['season'] == VALID_SEASON]
model_cv = fit_run_value_model(decisive_table, TRAIN_SEASONS)
pred_valid = model_cv.predict(valid_set[MODEL_FEATURES])
model_r2 = r2_score(valid_set['woba_value'], pred_valid)
print(f'(전체 피처) GBM 검증 R²: {model_r2:.4f}')
assert model_r2 > 0

final_model = fit_run_value_model(decisive_table, decisive_table['season'].unique())
print('(전체 피처) 최종 모델 준비 완료 -- 선택가치 계산에 씀')

(전체 피처) GBM 검증 R²: 0.0568


(전체 피처) 최종 모델 준비 완료 -- 선택가치 계산에 씀


In [7]:
def find_rematch_outcome_pitch(xbh_: pd.DataFrame, pitches_sit: pd.DataFrame) -> pd.DataFrame:
    decisive = pitches_sit[pitches_sit['woba_value'].notna()][
        ['game_pk', 'pitcher', 'season', 'at_bat_number', *MODEL_FEATURES, 'platoon_match', 'events', 'woba_value']
    ]
    keys = xbh_[['game_pk', 'pitcher', 'next_at_bat_number', 'reused_same_type']].rename(
        columns={'next_at_bat_number': 'at_bat_number'}
    )
    out = keys.merge(decisive, on=['game_pk', 'pitcher', 'at_bat_number'], how='left', validate='many_to_one')
    return out


REQUIRED_OUTCOME_COLUMNS = sorted(set(['events', 'woba_value', 'platoon_match', *MODEL_FEATURES]))


def build_avoidance_outcome_dataset(found: pd.DataFrame) -> pd.DataFrame:
    out = found.copy()
    out['pitch_family'] = out['pitch_type'].map(map_pitch_family)
    out = out.dropna(subset=[*REQUIRED_OUTCOME_COLUMNS, 'pitch_family', 'reused_same_type'])
    out['avoided'] = (1 - out['reused_same_type']).astype(int)
    out['allowed_xbh_again'] = out['events'].isin(EXTRA_BASE_HIT_EVENTS).astype(int)
    out['pitcher'] = out['pitcher'].astype(str)
    return out


found = find_rematch_outcome_pitch(xbh, decisive_table)
found_with_pa = found.dropna(subset=['woba_value'])
model_data = build_avoidance_outcome_dataset(found_with_pa)
assert len(model_data) == 27_245, len(model_data)
print(f'모델링에 쓸 재대결: {len(model_data):,}건, 투수 {model_data["pitcher"].nunique()}명 (노트북 09와 동일해야 함)')

모델링에 쓸 재대결: 27,245건, 투수 695명 (노트북 09와 동일해야 함)


In [8]:
def attach_counterfactual_pitch_type(model_data_: pd.DataFrame, xbh_: pd.DataFrame, season_usage_: pd.DataFrame) -> pd.DataFrame:
    cf_keys = xbh_[['game_pk', 'pitcher', 'next_at_bat_number', 'hit_pitch_type']].rename(
        columns={'next_at_bat_number': 'at_bat_number'}
    )
    cf_keys['pitcher'] = cf_keys['pitcher'].astype(str)
    out = model_data_.merge(cf_keys, on=['game_pk', 'pitcher', 'at_bat_number'], how='left', validate='many_to_one')

    usage = season_usage_.rename(
        columns={'pitch_type': 'hit_pitch_type', 'season_usage_rate': 'hit_pitch_season_usage_rate'}
    ).copy()
    usage['pitcher'] = usage['pitcher'].astype(str)
    out = out.merge(
        usage[['pitcher', 'season', 'hit_pitch_type', 'hit_pitch_season_usage_rate']],
        on=['pitcher', 'season', 'hit_pitch_type'], how='left',
    )
    out['hit_pitch_season_usage_rate'] = out['hit_pitch_season_usage_rate'].fillna(0.0)
    return out


model_data = attach_counterfactual_pitch_type(model_data, xbh, season_usage)
assert model_data['hit_pitch_type'].notna().all()
print('반사실 구종 결측 0건 확인')

반사실 구종 결측 0건 확인


In [9]:
def compute_selection_value(df: pd.DataFrame, model) -> pd.Series:
    actual_pred = model.predict(df[MODEL_FEATURES])
    cf = df[MODEL_FEATURES].copy()
    cf['pitch_type'] = df['hit_pitch_type'].astype(df['pitch_type'].dtype)
    cf['season_usage_rate'] = df['hit_pitch_season_usage_rate']
    cf_pred = model.predict(cf)
    return pd.Series(cf_pred - actual_pred, index=df.index)


class _LinearStub:
    def predict(self, X):
        return np.where(X['pitch_type'].astype(str) == 'SL', 0.5, 0.2)


_cat_dtype = pd.CategoricalDtype(categories=['SL', 'FF', 'CH'])
_df_sv = pd.DataFrame({
    'pitch_type': pd.Categorical(['FF', 'SL'], dtype=_cat_dtype), 'hit_pitch_type': ['SL', 'SL'],
    'season_usage_rate': [0.4, 0.5], 'hit_pitch_season_usage_rate': [0.3, 0.5],
    'stand': ['R', 'R'], 'p_throws': ['R', 'R'], 'balls': [0, 0], 'strikes': [0, 0],
    'outs_when_up': [0, 0], 'runners_n': [0, 0], 'risp': [0, 0], 'score_diff': [0, 0],
})
_sv = compute_selection_value(_df_sv, _LinearStub())
assert np.allclose(_sv.to_numpy(), [0.3, 0.0])
print('compute_selection_value 테스트 통과')

compute_selection_value 테스트 통과


In [10]:
def build_rematch_pa_pitches(xbh_: pd.DataFrame, pitches_sit: pd.DataFrame, season_usage_: pd.DataFrame, pitch_type_dtype) -> pd.DataFrame:
    keys = xbh_[['game_pk', 'pitcher', 'season', 'next_at_bat_number', 'hit_pitch_type']].rename(
        columns={'next_at_bat_number': 'at_bat_number'}
    )
    cols = ['game_pk', 'pitcher', 'at_bat_number', 'pitch_type', 'stand', 'p_throws',
            'balls', 'strikes', 'outs_when_up', 'runners_n', 'risp', 'score_diff']
    out = keys.merge(pitches_sit[cols], on=['game_pk', 'pitcher', 'at_bat_number'], how='left')
    out = out.dropna(subset=['pitch_type'])

    out = out.merge(
        season_usage_[['pitcher', 'season', 'pitch_type', 'season_usage_rate']],
        on=['pitcher', 'season', 'pitch_type'], how='left',
    )
    usage = season_usage_.rename(
        columns={'pitch_type': 'hit_pitch_type', 'season_usage_rate': 'hit_pitch_season_usage_rate'}
    )
    out = out.merge(
        usage[['pitcher', 'season', 'hit_pitch_type', 'hit_pitch_season_usage_rate']],
        on=['pitcher', 'season', 'hit_pitch_type'], how='left',
    )
    out['hit_pitch_season_usage_rate'] = out['hit_pitch_season_usage_rate'].fillna(0.0)
    out = out.dropna(subset=['season_usage_rate'])
    out['pitch_type'] = out['pitch_type'].astype(pitch_type_dtype)
    out['pitcher'] = out['pitcher'].astype(str)
    for col in ('stand', 'p_throws'):
        out[col] = out[col].astype('category')
    return out


def aggregate_pa_selection_value(pa_pitches: pd.DataFrame, model) -> pd.Series:
    sv = compute_selection_value(pa_pitches, model)
    return pa_pitches.assign(_sv=sv).groupby(['game_pk', 'pitcher', 'at_bat_number'])['_sv'].mean()


_TRAINED_DTYPE = pd.CategoricalDtype(categories=['SL', 'FF', 'CU', 'CH'])
_xbh_pa = pd.DataFrame({
    'game_pk': [100], 'pitcher': [10], 'season': [2021], 'next_at_bat_number': [5.0], 'hit_pitch_type': ['SL'],
})
_px_pa = pd.DataFrame({
    'game_pk': [100, 100, 100, 100], 'pitcher': [10, 10, 10, 10], 'at_bat_number': [5, 5, 5, 9],
    'pitch_type': ['FF', 'CU', 'FF', 'SL'], 'stand': ['R'] * 4, 'p_throws': ['R'] * 4,
    'balls': [0, 1, 2, 0], 'strikes': [0, 1, 2, 0], 'outs_when_up': [0, 0, 0, 0],
    'runners_n': [0, 0, 0, 0], 'risp': [0, 0, 0, 0], 'score_diff': [0, 0, 0, 0],
})
_su_pa = pd.DataFrame({
    'pitcher': [10, 10, 10], 'season': [2021, 2021, 2021], 'pitch_type': ['FF', 'CU', 'SL'],
    'season_usage_rate': [0.5, 0.3, 0.2],
})
_pa_pitches = build_rematch_pa_pitches(_xbh_pa, _px_pa, _su_pa, _TRAINED_DTYPE)
assert len(_pa_pitches) == 3
assert _pa_pitches['pitch_type'].astype(str).tolist() == ['FF', 'CU', 'FF']
print('build_rematch_pa_pitches 테스트 통과')

_pa_full_avoid = pd.DataFrame({
    'game_pk': [100] * 3, 'pitcher': ['10'] * 3, 'at_bat_number': [5] * 3,
    'pitch_type': pd.Categorical(['FF', 'CU', 'FF'], dtype=_TRAINED_DTYPE), 'hit_pitch_type': ['SL'] * 3,
    'season_usage_rate': [0.5, 0.3, 0.5], 'hit_pitch_season_usage_rate': [0.2, 0.2, 0.2],
    'stand': ['R'] * 3, 'p_throws': ['R'] * 3, 'balls': [0, 1, 2], 'strikes': [0, 1, 2],
    'outs_when_up': [0, 0, 0], 'runners_n': [0, 0, 0], 'risp': [0, 0, 0], 'score_diff': [0, 0, 0],
})
_out_a = aggregate_pa_selection_value(_pa_full_avoid, _LinearStub())
assert np.isclose(_out_a.loc[(100, '10', 5)], 0.3)
print('aggregate_pa_selection_value 테스트 통과')

build_rematch_pa_pitches 테스트 통과
aggregate_pa_selection_value 테스트 통과


In [11]:
pa_pitches = build_rematch_pa_pitches(xbh, pitches, season_usage, decisive_table['pitch_type'].dtype)
pa_mean = aggregate_pa_selection_value(pa_pitches, final_model).rename('selection_value_pa_mean').reset_index()
model_data = model_data.merge(pa_mean, on=['game_pk', 'pitcher', 'at_bat_number'], how='left', validate='one_to_one')
assert model_data['selection_value_pa_mean'].notna().all()
print(f'노트북 09와 동일한 전체 표본 준비 완료: {len(model_data):,}건, 투수 {model_data["pitcher"].nunique()}명')

노트북 09와 동일한 전체 표본 준비 완료: 27,245건, 투수 695명


## 3. 패스트볼 계열 제외

노트북 10이 "주무기" 후보에서 패스트볼 계열을 뺀 것과 같은 이유입니다 — 대부분 투수의 최다 사용
구종은 패스트볼이고, 패스트볼은 카운트를 잡으려고 구조적으로 계속 던져야 하는 구종이라 "맞은 구종을
피한다"는 선택이 변화구를 피하는 것과 의미가 다릅니다. 맞은 구종(`hit_pitch_type`)이 패스트볼
계열이었던 재대결은 이 상호작용 분석에서 제외합니다 — `hit_pitch_season_usage_rate`가 이제부터는
항상 "맞은 비-패스트볼 구종의 그 시즌 구사율"을 가리킵니다.

In [12]:
model_data['hit_pitch_family'] = model_data['hit_pitch_type'].map(map_pitch_family)
n_before_filter = len(model_data)
model_data = model_data[model_data['hit_pitch_family'] != 'fastball'].copy()
print(f'패스트볼 계열을 맞은 재대결 제외: {n_before_filter:,}건 -> {len(model_data):,}건 '
      f'(투수 {model_data["pitcher"].nunique()}명)')

패스트볼 계열을 맞은 재대결 제외: 27,245건 -> 10,747건 (투수 616명)


## 4. 상호작용 모델

`hit_pitch_season_usage_rate`(맞은 구종의 그 시즌 구사율)를 표본 평균 중심으로 센터링하고,
`avoided * hit_pitch_season_usage_rate_c`를 공식에 넣습니다. 중심화하면 `avoided`의 주효과가
"구사율이 평균일 때의 회피 효과"로 해석되고, 상호작용항은 "구사율이 1단위(=100%p) 올라갈 때마다
회피 효과가 얼마나 바뀌는가"를 뜻합니다.

In [13]:
model_data['hit_pitch_season_usage_rate_c'] = (
    model_data['hit_pitch_season_usage_rate'] - model_data['hit_pitch_season_usage_rate'].mean()
)

FORMULA_INTERACTION = (
    'selection_value_pa_mean ~ avoided * hit_pitch_season_usage_rate_c + outs_when_up + score_diff '
    '+ platoon_match + risp + runners_n + factor(season) + (1 | pitcher)'
)
fit_lmer(model_data, FORMULA_INTERACTION)
print('=== 수렴 상태 ===')
print(check_convergence())
fixed_interaction = extract_lmer_fixed_effects()
print('=== 고정효과 (avoided, 상호작용항 포함) ===')
display(fixed_interaction.round(5))
vc_interaction = extract_variance_components('pitcher')
print('분산 성분:', vc_interaction)
if vc_interaction['pitcher_intercept'] is None or vc_interaction['residual'] is None:
    icc_interaction = None
    print('ICC 계산 불가')
else:
    icc_interaction = compute_icc_gaussian(vc_interaction['pitcher_intercept'], vc_interaction['residual'])
    print(f'ICC: {icc_interaction:.4f}')

2026-10-06 12:18:21,711 [WARNING] R callback write-console: Loading required package: Matrix
  


=== 수렴 상태 ===
OK: no convergence warnings; isSingular=False
=== 고정효과 (avoided, 상호작용항 포함) ===


,term,estimate,std_error,t_value,ci_low,ci_high,p_value
0,(Intercept),-0.00853,0.00088,-9.70871,-0.01025,-0.00681,0.00000
1,avoided,0.00286,0.00053,5.44829,0.00183,0.00389,0.00000
2,hit_pitch_season_usage_rate_c,-0.06699,0.00468,-14.31437,-0.07616,-0.05781,0.00000
3,outs_when_up,-0.00063,0.00029,-2.17730,-0.00120,-0.00006,0.02946
4,score_diff,-0.00015,0.00008,-1.88225,-0.00030,0.00001,0.05980
5,platoon_match,0.00669,0.00049,13.69892,0.00573,0.00764,0.00000
6,risp,-0.00030,0.00085,-0.35462,-0.00196,0.00136,0.72287
7,runners_n,0.00266,0.00048,5.54236,0.00172,0.00360,0.00000
8,factor(season)2022,0.00101,0.00084,1.19931,-0.00064,0.00265,0.23041
9,factor(season)2023,0.00060,0.00083,0.72375,-0.00103,0.00223,0.46922


분산 성분: {'pitcher_intercept': 5.187227811785932e-05, 'residual': 0.0005556767373730973}
ICC: 0.0854


## 5. 견고성 검증: GBM 재적합 불확실성 (상호작용항)

노트북 09·10의 (C) 검증과 같은 방법입니다. 다만 매 부트스트랩마다 `lmer`를 다시 적합하는 건 비용이
크므로, 고정효과만 보는 빠른 OLS로 대체합니다 — ICC가 작아(09·10에서 각각 0.003, 0.33이었지만
고정효과 자체는 랜덤효과 유무와 거의 안 바뀌는 패턴을 반복해서 확인했으므로) 상호작용 계수의
부트스트랩 변동성을 보는 용도로는 충분한 근사입니다.

In [14]:
OLS_FORMULA = (
    '{value_col} ~ avoided * hit_pitch_season_usage_rate_c + outs_when_up + score_diff + platoon_match '
    '+ risp + runners_n + C(season)'
)


def compute_interaction_coefficient_ols(df: pd.DataFrame, value_col: str) -> float:
    model = smf.ols(OLS_FORMULA.format(value_col=value_col), data=df).fit()
    return float(model.params['avoided:hit_pitch_season_usage_rate_c'])


def gbm_refit_bootstrap_interaction_sd(decisive_table_, pa_pitches_, model_data_, seasons, n_boot, random_state):
    rng = np.random.default_rng(random_state)
    train = decisive_table_[decisive_table_['season'].isin(seasons)]
    coefs = []
    for b in range(n_boot):
        resample_idx = rng.integers(0, len(train), size=len(train))
        boot_train = train.iloc[resample_idx]
        model = HistGradientBoostingRegressor(
            categorical_features='from_dtype', random_state=random_state + b, max_iter=200, early_stopping=True,
        )
        model.fit(boot_train[MODEL_FEATURES], boot_train['woba_value'])
        pa_mean = aggregate_pa_selection_value(pa_pitches_, model).rename('sv_boot').reset_index()
        merged = model_data_.merge(pa_mean, on=['game_pk', 'pitcher', 'at_bat_number'])
        coefs.append(compute_interaction_coefficient_ols(merged, 'sv_boot'))
    return coefs

In [15]:
def _make_synthetic_df(n, rng, true_coef, value_col='selection_value_pa_mean'):
    avoided = rng.integers(0, 2, n)
    usage_c = rng.uniform(-0.3, 0.3, n)
    df = pd.DataFrame({
        'avoided': avoided, 'hit_pitch_season_usage_rate_c': usage_c,
        'outs_when_up': rng.integers(0, 3, n), 'score_diff': rng.integers(-3, 3, n),
        'platoon_match': rng.integers(0, 2, n).astype(float), 'risp': rng.integers(0, 2, n),
        'runners_n': rng.integers(0, 4, n), 'season': rng.choice([2021, 2022, 2023], size=n),
    })
    df[value_col] = true_coef * df['avoided'] * df['hit_pitch_season_usage_rate_c']
    return df


_rng_t = np.random.default_rng(0)
_df_t = _make_synthetic_df(400, _rng_t, true_coef=2.0)
_coef_t = compute_interaction_coefficient_ols(_df_t, 'selection_value_pa_mean')
assert np.isclose(_coef_t, 2.0, atol=1e-6)
print('compute_interaction_coefficient_ols 테스트 통과')

compute_interaction_coefficient_ols 테스트 통과


In [16]:
_lmer_coef = float(fixed_interaction.loc[fixed_interaction['term'] == 'avoided:hit_pitch_season_usage_rate_c', 'estimate'].iloc[0])
_ols_coef_check = compute_interaction_coefficient_ols(model_data, 'selection_value_pa_mean')
print(f'lmer 상호작용 계수: {_lmer_coef:.5f}, OLS(고정효과만) 근사치: {_ols_coef_check:.5f} '
      f'(랜덤효과 유무로 거의 안 바뀌어야 OLS 근사가 타당함)')

N_BOOT = 50
boot_coefs = gbm_refit_bootstrap_interaction_sd(
    decisive_table, pa_pitches, model_data, decisive_table['season'].unique(), N_BOOT, RANDOM_STATE,
)
boot_sd = float(np.std(boot_coefs))
_lmer_se = float(fixed_interaction.loc[fixed_interaction['term'] == 'avoided:hit_pitch_season_usage_rate_c', 'std_error'].iloc[0])
_combined_se = float(np.sqrt(_lmer_se ** 2 + boot_sd ** 2))
print(f'GBM 재적합 부트스트랩 {N_BOOT}회: 상호작용 계수의 표준편차 = {boot_sd:.5f}')
print(f'lmer가 보고한 SE = {_lmer_se:.5f}')
print(f'두 불확실성을 합친(제곱합 제곱근) 보수적 SE = {_combined_se:.5f}')

_t_combined = _lmer_coef / _combined_se
from scipy import stats as _stats
_p_combined = float(2 * (1 - _stats.norm.cdf(abs(_t_combined))))
print(f'보수적 SE 기준 t={_t_combined:.2f}, p={_p_combined:.2e} (여전히 매우 유의)')

lmer 상호작용 계수: -0.06446, OLS(고정효과만) 근사치: -0.07330 (랜덤효과 유무로 거의 안 바뀌어야 OLS 근사가 타당함)


GBM 재적합 부트스트랩 50회: 상호작용 계수의 표준편차 = 0.01473
lmer가 보고한 SE = 0.00531
두 불확실성을 합친(제곱합 제곱근) 보수적 SE = 0.01566
보수적 SE 기준 t=-4.12, p=3.84e-05 (여전히 매우 유의)


## 6. 구사율별 예측 효과 — 구체적으로 어느 지점에서 뒤집히는가

In [17]:
_avoided_main = float(fixed_interaction.loc[fixed_interaction['term'] == 'avoided', 'estimate'].iloc[0])
_mean_usage = float(model_data['hit_pitch_season_usage_rate'].mean())
print(f'맞은 구종의 평균 구사율: {_mean_usage:.3f}')
print('구사율별 예측 avoided 효과(평균 구사율 기준 중심화 계수 사용):')
for usage in [0.02, 0.05, 0.10, 0.15, 0.20, 0.30, 0.40]:
    usage_c = usage - _mean_usage
    predicted = _avoided_main + _lmer_coef * usage_c
    print(f'  구사율 {usage:.2f}: 예측 avoided 효과 = {predicted:+.5f}')

맞은 구종의 평균 구사율: 0.213
구사율별 예측 avoided 효과(평균 구사율 기준 중심화 계수 사용):
  구사율 0.02: 예측 avoided 효과 = +0.01531
  구사율 0.05: 예측 avoided 효과 = +0.01337
  구사율 0.10: 예측 avoided 효과 = +0.01015
  구사율 0.15: 예측 avoided 효과 = +0.00693
  구사율 0.20: 예측 avoided 효과 = +0.00370
  구사율 0.30: 예측 avoided 효과 = -0.00274
  구사율 0.40: 예측 avoided 효과 = -0.00919


## 7. 해석과 한계

* **표본.** 패스트볼 계열을 맞은 재대결을 뺀 결과, 27,245건 중 **10,747건(39.4%), 투수 616명**이
  남았습니다.
* **상호작용이 매우 강하게, 그리고 가설과 반대 방향으로 유의합니다.** `avoided:hit_pitch_season_usage_rate_c`
  계수는 **−0.06446**, t=**−12.14**, p≈0 입니다. GBM 재적합 불확실성을 반영한 보수적 SE(0.01566,
  `lmer` 자체 SE의 약 3배)로 다시 계산해도 t=−4.12, **p≈3.8×10⁻⁵**로 여전히 매우 유의합니다.
* **구사율에 따라 효과가 선형적으로 줄어들다가 뒤집힙니다.** 맞은 구종(이제 패스트볼 제외)의 평균
  구사율은 0.213입니다. 6절의 예측값을 보면:
  | 구사율 | 예측 avoided 효과 |
  |---|---|
  | 0.02 (거의 안 쓰는 구종) | **+0.0153** |
  | 0.10 | +0.0102 |
  | 0.20 (평균 근접) | +0.0037 |
  | 0.30 | **−0.0027** |
  | 0.40 (뚜렷한 주무기) | **−0.0092** |

  패스트볼을 포함했던 이전 버전(교차점 ≈0.30, 구사율 0.40에서 −0.0036)보다 **교차점이 더 낮아지고
  (≈0.25), 양 끝의 크기도 더 커졌습니다**(+0.0153 / −0.0092) — 패스트볼을 섞었을 때는 "계속 던져야
  하는 패스트볼"과 "진짜 주무기 변화구"가 한 변수에 뭉개져 효과가 희석됐다는 뜻입니다. 패스트볼을
  제외하고 보니 "거의 안 쓰는 구종을 피하는 이득"과 "주무기를 버리는 손해"가 둘 다 더 뚜렷해졌습니다.
* **야구적으로 말이 됩니다.** 거의 안 쓰는 구종에 맞았다면 다른 구종으로 바꾸는 게 합리적인 선택일
  가능성이 높습니다(그 구종이 애초에 약점이었을 수 있음). 반대로 주무기는 그 투수가 가장 많이,
  가장 자주 쓰는 데는 이유가 있는 구종이라, 한 타석 맞았다고 버리는 게 오히려 비합리적 선택(평균
  회귀를 무시한 과잉 반응)일 수 있습니다.
* **노트북 09·10과의 관계.** 이 노트북은 09·10의 결과를 수정하지 않습니다. 09는 패스트볼을 포함한
  전체 표본의 평균(+0.0023)이고, 10은 패스트볼을 뺀 주무기(상위 1개 구종)만 본 이분법적 부분표본
  (효과 없음)입니다. 이 노트북은 10과 같은 패스트볼 제외 기준을 쓰면서도 표본을 자르지 않아, "주무기
  일수록 효과가 사라지다가 역전된다"는 연속적인 그림을 가장 분명하게 보여줍니다.
* **ICC.** 0.0854로, 09(0.0041, 패스트볼 포함)보다는 높고 10(0.33, 상위 1개 구종만)보다는 낮습니다 —
  표본을 좁힐수록(패스트볼 제외 → 주무기 1개만) 투수 간 차이가 더 크게 드러나는 일관된 패턴입니다.
* **여전히 남은 한계.** 선택편향(관측되지 않는 공통원인, `pitcher_id`를 뺀 설계상 해결 불가능 —
  특히 "주무기가 좋은 투수일수록 그 구종에 더 의존한다"는 구조 자체가 구사율과 얽혀 있을 수 있음),
  GBM의 `pitch_type`/구사율 효과가 매치업이 아니라 리그 평균 수준이라는 점, 다중비교 미보정(09·10·11
  세 분석 모두 포함), `season_usage_rate`의 시즌 전체 집계가 미래 정보를 포함한다는 점은 그대로
  적용됩니다.